# Python and NumPy foundations for neural networks

*Neural Networks from Scratch: Build, Train and Understand AI Models · Lesson 1*

A neural network is a few lines of arithmetic repeated many times over arrays of numbers. Before you build one, this notebook makes the tools for that arithmetic feel natural: Python functions and classes, NumPy arrays and their shapes, broadcasting, matrix multiplication, random numbers you can reproduce, feature scaling, and the dataset you will use for the rest of the course.

## What you will build

You will type, test and use four functions of your own:

| Function | What it does | Where it is used again |
|---|---|---|
| `dot_loop(u, v)` | a dot product written as a loop | inside every neuron, from Lesson 2 |
| `matmul_loops(A, B)` | matrix multiplication with three loops and shape checks | this lesson, to see why NumPy's `@` matters |
| `standardize(X)` | rescales every feature to mean 0 and standard deviation 1 | preparing data in later lessons |
| `make_spirals(n_per_class, n_classes, noise, rng)` | the spiral dataset of this course | every later lesson |

## Learning objectives

By the end of this notebook you will be able to:

1. write Python functions and small classes, and explain why this course stores layers in dictionaries;
2. explain how a NumPy array differs from a Python list, and read the shape of any array;
3. predict the result of broadcasting, and spot the shape bug it can hide;
4. multiply matrices by hand, with loops and with `@`, and explain the difference in speed;
5. create reproducible random numbers with `np.random.default_rng`, and choose a sensible scale for random weights;
6. standardise features and explain why it helps a neural network;
7. generate, plot and inspect the spiral dataset.

## Dependencies

**Tier 1: NumPy and Matplotlib.** The core cells use only the Python standard library, NumPy and Matplotlib. One optional cell in section 9, tagged `data`, shows the dataset as a pandas table; nothing after it depends on it, and it prints a short note instead if pandas is not installed.

The notebook runs without installing anything in JupyterLite (in your browser), on your Furina JupyterHub server and in Google Colab. It needs no internet access and runs from top to bottom in a few seconds, or a little longer in the browser.

### How the type-along cells work

Each building block follows the same rhythm: a short explanation with the maths and the shape of every array, a small demonstration to run, then **✍️ Your turn**: a specification of a function for you to type into the cell provided. The check cell after it compares your function with a reference version on a few test inputs and prints one of three symbols:

- ✍️ means you have not written the function yet. The notebook carries on with the reference version.
- ❌ means your function ran but returned something different, or raised an error. The message shows the input, your result and the expected result, and the notebook carries on with the reference version until yours matches.
- ✅ means your function matches. **From then on, the notebook runs your code.**

The reference versions sit in collapsed cells, which JupyterLab and JupyterLite show as a thin bar with "…"; click it to expand the cell. Some other notebook apps show them expanded, so scroll past if you want to work things out yourself. Because every check falls back to a working version, **Run All** always works, even before you have typed anything.

Cells marked **🧪 Try it** are experiments: change the number the comment points to and run the cell again.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import time

import matplotlib.pyplot as plt

## 1. Python functions and classes

A **function** packages a calculation behind a name. It takes inputs (its *arguments*), computes something and `return`s an output. The first line inside it can be a *docstring*: a sentence that says what goes in and what comes out, including the shapes of any arrays. When an input is wrong, a good function says so loudly with `raise ValueError(...)` instead of quietly returning a wrong answer.

A **class** bundles data (*attributes*) with the functions that use it (*methods*). You create objects from a class, and each object keeps its own state between calls; `__init__` sets that state up when the object is created.

This course keeps the arithmetic in small functions and stores a network's parameters in plain dictionaries and lists rather than in classes: **a layer is a dictionary `{"W": weights, "b": biases}` and a network is a list of layers.** Dictionaries print clearly and are easy to inspect, and small functions can be tested one at a time, which is exactly what the check cells do.

In [ ]:
def scale(values, factor=2.0):
    """Return a new list with every value multiplied by factor (2.0 unless you say otherwise)."""
    return [factor * value for value in values]


print(scale([1, 2, 3]))
print(scale([1, 2, 3], factor=0.5))


class RunningMean:
    """Keeps the mean of all the numbers added so far."""

    def __init__(self):
        self.total = 0.0
        self.count = 0

    def add(self, value):
        self.total += value
        self.count += 1

    def mean(self):
        return self.total / self.count


tracker = RunningMean()
for loss in [1.0, 0.75, 0.5]:
    tracker.add(loss)
print("mean so far:", tracker.mean(), "after", tracker.count, "values")

layer = {"W": [[0.5, -1.0], [2.0, 0.0]], "b": [0.1, -0.2]}   # the shape of a layer in this course
network = [layer, {"W": [[1.0], [1.0]], "b": [0.0]}]        # a network: a list of layers
print(len(network), "layers; the first layer's biases are", network[0]["b"])

### The dot product: the arithmetic inside every neuron

The **dot product** of two vectors of the same length $n$ multiplies them entry by entry and adds up the results:

$$u \cdot v = \sum_{i=1}^{n} u_i v_i = u_1 v_1 + u_2 v_2 + \dots + u_n v_n .$$

Both inputs have shape `(n,)` and the result is a single number. A neuron (Lesson 2) computes exactly this between its weights and its inputs, so it is worth writing once by hand.

In [ ]:
u = [1.0, 2.0, 3.0]
v = [4.0, -1.0, 0.5]
print(u[0] * v[0] + u[1] * v[1] + u[2] * v[2])   # 4 - 2 + 1.5, written out by hand

### ✍️ Your turn: `dot_loop`

Write `dot_loop(u, v)`:

- **Inputs:** `u` and `v`, two sequences of numbers of the same length `n`: Python lists or NumPy arrays of shape `(n,)`.
- **Output:** a Python `float`: the sum of `u[i] * v[i]` over every position `i`.
- **Shape check:** if the lengths differ, `raise ValueError` with a message that states both lengths.

Hints: check the lengths first; start a running total at `0.0`; loop with `for i in range(len(u))`; finish with `return float(total)`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The collapsed cell below holds a reference version, `dot_loop_reference`; open it only when you are stuck.

In [ ]:
def dot_loop_reference(u, v):
    """Dot product of two equally long sequences, computed with a loop."""
    if len(u) != len(v):
        raise ValueError(f"dot_loop needs two sequences of the same length, got {len(u)} and {len(v)}")
    total = 0.0
    for i in range(len(u)):
        total += u[i] * v[i]
    return float(total)

In [ ]:
def dot_loop(u, v):
    """Return the dot product of u and v (both of length n) as a float, computed with a loop."""
    # ✍️ Your code here (about six lines)
    raise NotImplementedError

In [ ]:
dot_loop = practice("dot_loop", dot_loop, dot_loop_reference,
                    cases=[(u, v),
                           (np.array([0.5, -2.0]), np.array([4.0, 1.0])),
                           ([7.0], [3.0]),
                           ([], [])])

**Expected behaviour.** ✅ once your loop matches. The first case is the one written out above, $4 - 2 + 1.5 = 3.5$. The last case is two empty vectors: a sum with no terms is 0, so your function should return `0.0` without complaint.

The next cell tests the error you wrote for vectors of different lengths. A function that refuses bad input saves hours later: in a network, a shape mistake that runs silently produces a model that learns from nonsense.

In [ ]:
try:
    result = dot_loop([1.0, 2.0, 3.0], [1.0, 2.0])
except ValueError as error:
    print("✅ dot_loop refuses vectors of different lengths:", error)
except Exception as error:
    print(f"❌ dot_loop raised {type(error).__name__} instead of ValueError: {error}")
else:
    print(f"❌ dot_loop returned {result} for lengths 3 and 2. Add a length check that raises ValueError.")

## 2. Lists versus NumPy arrays

A Python **list** can hold anything: numbers, strings, other lists, all mixed together. Arithmetic treats it as a container: `+` joins two lists and `* 2` repeats one. A NumPy **array** holds numbers of a single type (its `dtype`, usually `float64`) arranged in a grid with a fixed **shape**, and arithmetic works entry by entry. Arrays are what neural networks are made of: a batch of inputs, a matrix of weights, a vector of biases.

Arrays are also fast. A Python loop handles one number at a time, with Python's bookkeeping around every step, while a NumPy operation runs a single compiled loop over the whole array. Replacing a Python loop with whole-array operations is called **vectorisation**.

In [ ]:
values = [1.0, 2.0, 3.0]
array = np.array(values)

print(values + values, "   list + list joins")
print(array + array, "         array + array adds entry by entry")
print(values * 2, "   list * 2 repeats")
print(array * 2, "         array * 2 doubles every entry")
print("dtype:", array.dtype, "  shape:", array.shape)

The helper below measures how long a function takes. It runs the function `number` times in a row and divides, so even a very fast call gets a meaningful time, and it keeps the best of a few rounds, because other programs on your computer can slow down any single round.

In [ ]:
def best_time(function, *args, number=1, repeats=3):
    """Best time per call, in seconds, over `repeats` rounds of `number` calls."""
    best = float("inf")
    for _ in range(repeats):
        start = time.perf_counter()
        for _ in range(number):
            function(*args)
        best = min(best, (time.perf_counter() - start) / number)
    return max(best, 1e-9)  # browsers round their clocks; never report zero


def sum_of_squares_loop(numbers):
    total = 0.0
    for value in numbers:          # one number at a time, in Python
        total += value * value
    return total


def sum_of_squares_vectorised(numbers):
    return np.sum(numbers * numbers)   # one call; the loop runs inside NumPy


numbers = np.random.default_rng(0).normal(size=100_000)
loop_seconds = best_time(sum_of_squares_loop, numbers, number=1, repeats=1)
vector_seconds = best_time(sum_of_squares_vectorised, numbers, number=50)
print(f"Python loop: {loop_seconds * 1000:8.3f} ms   result {sum_of_squares_loop(numbers):.6f}")
print(f"vectorised:  {vector_seconds * 1000:8.3f} ms   result {sum_of_squares_vectorised(numbers):.6f}")
print(f"the vectorised version is roughly {loop_seconds / vector_seconds:,.0f} times faster")

## 3. Shapes and axes

Every array has a **shape**: a tuple with one number per dimension, or *axis*.

- Shape `(3,)` is a vector of 3 numbers (one axis).
- Shape `(4, 3)` is a matrix with 4 rows and 3 columns (two axes).
- Shapes `(4, 1)` and `(1, 3)` are matrices with a single column or a single row. They can hold the same numbers as a vector, but they behave differently, as section 4 shows.

In this course a batch of inputs is always called `X`, with shape `(batch, features)`: one row per example and one column per feature. Labels are `y`, with shape `(batch,)`: one whole number per example. Reductions such as `sum`, `mean` and `max` take an `axis` argument. `axis=0` collapses the rows and leaves one result per column (here, per feature); `axis=1` collapses the columns and leaves one result per row (here, per example).

In [ ]:
X_grid = np.arange(12.0).reshape(4, 3)    # 4 examples, 3 features
print(X_grid)
print("X_grid.shape             ", X_grid.shape)
print("X_grid[0].shape          ", X_grid[0].shape, "   one example")
print("X_grid[:, 0].shape       ", X_grid[:, 0].shape, "   one feature, as a vector")
print("X_grid[:, 0:1].shape     ", X_grid[:, 0:1].shape, " one feature, kept as a column")
print("X_grid.mean(axis=0)      ", X_grid.mean(axis=0), "   one mean per feature")
print("X_grid.sum(axis=1).shape ", X_grid.sum(axis=1).shape, "   one total per example")
print("X_grid.T.shape           ", X_grid.T.shape, " the transpose swaps the axes")

In [ ]:
# 🧪 Try it: predict each printed shape before you run the cell. Then change rows and columns and predict again.
rows, columns = 4, 3
M = np.ones((rows, columns))
print(M.reshape(-1).shape)                  # -1 means "whatever length makes it fit"
print(M.reshape(columns, rows).shape)
print(M.max(axis=1).shape)
print(M.max(axis=1, keepdims=True).shape)   # keepdims keeps the collapsed axis with size 1

## 4. Vectorisation and broadcasting

Entry-by-entry operations need arrays of matching shapes, but NumPy can also combine arrays of *different* shapes, by stretching one to fit the other without copying it. This is **broadcasting**. NumPy compares the two shapes from the right-hand end:

1. two sizes are compatible if they are equal or if one of them is 1;
2. a missing leading axis counts as size 1;
3. every axis of size 1 is stretched to match the other array.

Broadcasting is what makes a layer's formula $Z = XW + b$ work for a whole batch at once. `X @ W` has shape `(batch, n_out)` and the bias `b` has shape `(n_out,)`, so the same row of biases is added to every example:

$$\underbrace{(\text{batch},\, n_\text{out})}_{XW} + \underbrace{(n_\text{out},)}_{b} \;\longrightarrow\; (\text{batch},\, n_\text{out}).$$

In [ ]:
X_small = np.array([[1.0, 200.0],
                    [2.0, 300.0],
                    [3.0, 400.0]])     # (3, 2)
column_means = X_small.mean(axis=0)   # (2,)
print(X_small - column_means)         # (3, 2) - (2,) -> (3, 2): every column now averages 0

b_row = np.array([0.5, -0.5])         # (2,)
print(np.zeros((3, 2)) + b_row)       # the same bias row added to every example

### The broadcasting trap: `(n,)` against `(n, 1)`

Broadcasting never complains when shapes are compatible, even when the result is not what you meant. The classic accident is combining a vector of shape `(n,)` with a column of shape `(n, 1)`: both get stretched, and you get an `(n, n)` matrix instead of `n` numbers. Anything computed from that matrix, such as an average error, is a perfectly finite and perfectly wrong number.

In [ ]:
predictions = np.array([1.0, 2.0, 3.0])         # (3,)
targets = np.array([[1.0], [2.0], [4.0]])      # (3, 1): a column, perhaps sliced with X[:, 0:1]
difference = predictions - targets
print("shape of the difference:", difference.shape, " (expected (3,))")
print("wrong mean squared error:", np.mean(difference ** 2))
print("right mean squared error:", np.mean((predictions - targets.ravel()) ** 2))

## 5. Matrix multiplication

A layer of a neural network applies many neurons to many examples at once, and that is a matrix multiplication. If $A$ has shape $(n, m)$ and $B$ has shape $(m, p)$, their product $C = AB$ has shape $(n, p)$, and each entry is the dot product of a row of $A$ with a column of $B$:

$$C_{ij} = \sum_{k=1}^{m} A_{ik} B_{kj}.$$

The inner sizes must agree (both are $m$), and the outer sizes $n$ and $p$ give the shape of the result. In a layer, $A$ is the batch of inputs, `(batch, n_in)`; $B$ is the weight matrix, `(n_in, n_out)`; and the product holds every neuron's weighted sum for every example, `(batch, n_out)`.

In [ ]:
A = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])     # (2, 3)
B = np.array([[1.0, 0.0],
              [0.0, 1.0],
              [2.0, -1.0]])         # (3, 2)
print("C[0, 0] by hand:", 1.0 * 1.0 + 2.0 * 0.0 + 3.0 * 2.0)   # row 0 of A with column 0 of B
print(A @ B)                                                  # the whole (2, 2) product

### ✍️ Your turn: `matmul_loops`

Write `matmul_loops(A, B)` with three nested loops:

- **Inputs:** `A` with shape `(n, m)` and `B` with shape `(m, p)`, both 2-D NumPy arrays.
- **Output:** a new float array `C` with shape `(n, p)`, where `C[i, j]` is the sum over `k` of `A[i, k] * B[k, j]`.
- **Shape checks:** raise `ValueError` if either input is not 2-D (`A.ndim != 2`), or if the inner sizes differ (`A.shape[1] != B.shape[0]`). Put both shapes in the message.

Hints: unpack `n, m = A.shape` and `m2, p = B.shape`; create the result with `np.zeros((n, p))`; loop over `i`, then `j`, then `k`, adding each product into `C[i, j]`.

Type it rather than pasting it: typing is how the shapes and indices sink in. The collapsed cell below holds a reference version, `matmul_loops_reference`; open it only when you are stuck.

In [ ]:
def matmul_loops_reference(A, B):
    """Matrix product of A (n, m) and B (m, p), computed with three Python loops."""
    if A.ndim != 2 or B.ndim != 2:
        raise ValueError(f"matmul_loops needs two 2-D arrays, got shapes {A.shape} and {B.shape}")
    n, m = A.shape
    m2, p = B.shape
    if m != m2:
        raise ValueError(f"cannot multiply shapes {A.shape} and {B.shape}: inner sizes {m} and {m2} differ")
    C = np.zeros((n, p))
    for i in range(n):
        for j in range(p):
            for k in range(m):
                C[i, j] += A[i, k] * B[k, j]
    return C

In [ ]:
def matmul_loops(A, B):
    """Return A @ B for A (n, m) and B (m, p), computed with three loops; raise ValueError on bad shapes."""
    # ✍️ Your code here (about twelve lines)
    raise NotImplementedError

In [ ]:
check_rng = np.random.default_rng(1)
matmul_loops = practice("matmul_loops", matmul_loops, matmul_loops_reference,
                        cases=[(A, B),
                               (check_rng.normal(size=(4, 5)), check_rng.normal(size=(5, 3))),
                               (np.array([[2.0, -1.0, 0.5]]), np.array([[1.0], [4.0], [-2.0]])),  # (1, 3) @ (3, 1)
                               (np.array([[1.0], [2.0]]), np.array([[3.0, 4.0, 5.0]]))])        # (2, 1) @ (1, 3)

**Expected behaviour.** ✅ once your loops match. The last two cases deserve a second look. A `(1, 3)` row times a `(3, 1)` column is a `(1, 1)` matrix holding their dot product, while a `(2, 1)` column times a `(1, 3)` row is a full `(2, 3)` table of products, called an *outer product*. The rule "the inner sizes agree, the outer sizes give the result" covers both.

The next cell feeds your function two pairs of shapes that cannot be multiplied, and expects a `ValueError` for each.

In [ ]:
for left, right in [(np.ones((2, 3)), np.ones((2, 3))), (np.ones(3), np.ones((3, 2)))]:
    try:
        matmul_loops(left, right)
    except ValueError as error:
        print("✅ ValueError:", error)
    except Exception as error:
        print(f"❌ matmul_loops raised {type(error).__name__} instead of ValueError: {error}")
    else:
        print(f"❌ matmul_loops accepted shapes {left.shape} and {right.shape}. Add the shape checks.")

### Your loops against `@`

Your loops perform $n \times p \times m$ multiplications, one Python step at a time. NumPy's `@` performs the same multiplications in compiled code that also uses your processor's vector instructions. Time both on matrices the size of a small layer:

In [ ]:
timing_rng = np.random.default_rng(2)
A_layer = timing_rng.normal(size=(60, 40))    # 60 examples, 40 inputs
B_layer = timing_rng.normal(size=(40, 30))    # 40 inputs, 30 neurons

loop_seconds = best_time(matmul_loops, A_layer, B_layer, number=1, repeats=1)
numpy_seconds = best_time(np.matmul, A_layer, B_layer, number=200)
print(f"three loops: {loop_seconds * 1000:9.3f} ms")
print(f"A @ B:       {numpy_seconds * 1000:9.3f} ms")
print(f"@ is roughly {loop_seconds / numpy_seconds:,.0f} times faster on this example")
print("same answer:", np.allclose(matmul_loops(A_layer, B_layer), A_layer @ B_layer))

In [ ]:
# 🧪 Try it: set size to 10, 20 and 40 in turn and run the cell each time.
# Doubling size doubles n, m and p, so the loops do 2 x 2 x 2 = 8 times as much work.
# (Keep size at 60 or below in the browser, where Python runs more slowly.)
size = 20
square = np.ones((size, size))
seconds = best_time(matmul_loops, square, square, number=1, repeats=1)
print(f"({size}, {size}) @ ({size}, {size}): {size ** 3:,} multiplications in {seconds * 1000:.1f} ms")

## 6. Random numbers you can reproduce

A network starts from random weights, and training shuffles the data, so randomness is everywhere in deep learning. To make an experiment repeatable, create a **generator** with a fixed *seed*, `rng = np.random.default_rng(42)`. A generator produces the same sequence of numbers every time it is created with the same seed. Each draw moves it along that sequence, so two draws from one generator differ, while two generators with the same seed agree.

Pass a generator into every function that needs randomness, as an argument called `rng`, instead of relying on a hidden global one. Then a single seed controls the whole experiment.

Useful methods: `rng.normal(loc, scale, size)` draws from a normal (Gaussian) distribution with mean `loc` and standard deviation `scale`; `rng.uniform(low, high, size)` draws evenly between two limits; `rng.integers(low, high, size)` draws whole numbers from `low` up to, but not including, `high`; `rng.permutation(n)` shuffles `0, 1, …, n - 1`.

In [ ]:
first = np.random.default_rng(42)
second = np.random.default_rng(42)
print(first.normal(0.0, 1.0, size=3))
print(second.normal(0.0, 1.0, size=3), "  same seed, same numbers")
print(first.normal(0.0, 1.0, size=3), "  the next draw from the same generator moves on")
print(np.random.default_rng(7).integers(0, 10, size=5), np.random.default_rng(7).permutation(5))

### Why the scale of random weights matters

A neuron adds up $n_\text{in}$ products of inputs and weights. If the inputs have roughly unit size and each weight has standard deviation $\sigma$, the sum has a standard deviation of about $\sigma\sqrt{n_\text{in}}$. With $\sigma = 1$, a neuron with 1,000 inputs produces numbers about 30 times larger than its inputs, and a few such layers in a row make the numbers explode. Choosing $\sigma = \sqrt{1/n_\text{in}}$ keeps the output at the scale of the input. Lesson 3 uses $\sigma = \sqrt{2/n_\text{in}}$, called *He initialisation*, which suits the ReLU activation you will meet in Lesson 2.

In [ ]:
# 🧪 Try it: change weight_scale to "1/n_in" and then to "2/n_in", and run the cell again.
weight_scale = "1"          # "1", "1/n_in" or "2/n_in"

experiment_rng = np.random.default_rng(3)
for n_in in [2, 16, 128, 1024]:
    inputs = experiment_rng.normal(0.0, 1.0, size=(500, n_in))       # 500 examples, unit-scale features
    sigma = {"1": 1.0, "1/n_in": np.sqrt(1 / n_in), "2/n_in": np.sqrt(2 / n_in)}[weight_scale]
    weights = experiment_rng.normal(0.0, sigma, size=(n_in, 8))       # 8 neurons
    outputs = inputs @ weights
    print(f"n_in = {n_in:5d}: weight std {sigma:.3f} -> output std {outputs.std():7.3f}")

## 7. Standardising features

Real features arrive on very different scales: a height in centimetres is around 170, a probability lies between 0 and 1, and an income may be in the tens of thousands. A neuron's weighted sum is then dominated by the feature with the largest numbers, and the starting weights from section 6, which assume inputs of roughly unit size, no longer fit. **Standardising** gives every feature (every column of `X`) mean 0 and standard deviation 1:

$$x'_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}, \qquad \mu_j = \frac{1}{N}\sum_{i=1}^{N} x_{ij}, \qquad \sigma_j = \sqrt{\frac{1}{N}\sum_{i=1}^{N} \left(x_{ij} - \mu_j\right)^2}.$$

Shapes: `X` is `(batch, features)`, the means $\mu$ and standard deviations $\sigma$ are `(features,)`, and broadcasting does the rest. Keep $\mu$ and $\sigma$: new data must be scaled with the *same* numbers as the training data, never with statistics of its own.

In [ ]:
people = np.array([[160.0, 0.2],
                   [175.0, 0.9],
                   [190.0, 0.4]])       # height in cm, and a probability
print("means:", people.mean(axis=0), "  standard deviations:", people.std(axis=0))
print(people - people.mean(axis=0))      # centred: every column now averages 0

### ✍️ Your turn: `standardize`

Write `standardize(X)`:

- **Input:** `X`, a float array with shape `(batch, features)`.
- **Output:** a tuple `(X_std, mean, std)`, where `mean = X.mean(axis=0)` and `std = X.std(axis=0)` both have shape `(features,)`, and `X_std = (X - mean) / std` has the shape of `X`. NumPy's `std` divides by $N$ by default, as in the formula above.
- **Constant columns:** a column whose values are all equal has standard deviation 0, and dividing by it gives NaN ("not a number"). Replace each zero in `std` with `1.0` *before* dividing, and return that adjusted `std`. Such a column then becomes a column of zeros.

Hint: `np.where(std == 0, 1.0, std)` returns a copy of `std` with its zeros replaced by 1.0.

Type it rather than pasting it: typing is how the shapes and indices sink in. The collapsed cell below holds a reference version, `standardize_reference`; open it only when you are stuck.

In [ ]:
def standardize_reference(X):
    """Rescale every column of X (batch, features) to mean 0 and standard deviation 1.

    Returns (X_std, mean, std). A column with standard deviation 0 keeps std = 1.0,
    so it becomes a column of zeros instead of a column of NaNs.
    """
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    std = np.where(std == 0, 1.0, std)
    return (X - mean) / std, mean, std

In [ ]:
def standardize(X):
    """Return (X_std, mean, std) for X (batch, features); a zero std is replaced by 1.0."""
    # ✍️ Your code here (about four lines)
    raise NotImplementedError

In [ ]:
standardize = practice("standardize", standardize, standardize_reference,
                       cases=[(people,),
                              (np.random.default_rng(4).normal(5.0, 3.0, size=(50, 4)),),
                              (np.array([[1.0, 7.0], [2.0, 7.0], [3.0, 7.0]]),),   # a constant column
                              (np.array([[4.0], [8.0]]),)])                          # a single feature

**Expected behaviour.** ✅ once your three outputs match. Every column of `X_std` then has mean 0 and standard deviation 1, up to rounding, except constant columns, which become zeros. Because you return `mean` and `std`, the scaling can be undone with `X_std * std + mean`. The next cell checks both facts on the heights and probabilities.

In [ ]:
people_std, people_mean, people_sd = standardize(people)
print(np.round(people_std, 3))
print("column means:", np.round(people_std.mean(axis=0), 12), "  column stds:", np.round(people_std.std(axis=0), 12))
print("undoing the scaling recovers the data:", np.allclose(people_std * people_sd + people_mean, people))

## 8. A dataset to learn from: interleaved spirals

The rest of this course trains networks on a dataset you generate yourself: points in the plane arranged along spiral arms, one arm per class. No straight line separates the arms, so a model has to learn a curved boundary, which is exactly what a neural network can do and a single neuron cannot (Lesson 2 shows why).

With $K$ classes and $n$ points per class, point $i$ of class $k$ sits at radius $r_i$ and angle $\theta_{ik}$:

$$r_i = 0.05 + 0.95\,\frac{i}{n - 1}, \qquad \theta_{ik} = \frac{2\pi k}{K} + 4\, r_i + \varepsilon_{ik}, \qquad \varepsilon_{ik} \sim \mathcal{N}\!\left(0,\ \text{noise}^2\right),$$

$$\text{point} = \left(r_i \cos\theta_{ik},\ \ r_i \sin\theta_{ik}\right).$$

The radius grows evenly from 0.05 near the centre to 1.0 at the rim, and the angle grows with it, so each arm sweeps 4 radians (about 230°) as it winds outwards. Each class starts at its own angle, $2\pi k / K$, so the $K$ arms are evenly interleaved. The noise $\varepsilon$ jitters each angle, which smears points sideways across their arm; `noise = 0` gives perfect curves.

The result is `X` with shape `(n * K, 2)`, two coordinates per point, and `y` with shape `(n * K,)`, the class of each point.

In [ ]:
radius = np.linspace(0.05, 1.0, 5)            # five points from the centre to the rim
angle = 2 * np.pi * 0 / 3 + 4.0 * radius     # class 0 of 3, without noise
print(np.round(np.column_stack([radius * np.cos(angle), radius * np.sin(angle)]), 3))

### ✍️ Your turn: `make_spirals`

Write `make_spirals(n_per_class, n_classes, noise, rng)`:

- **Inputs:** `n_per_class` and `n_classes`, whole numbers; `noise`, the standard deviation of the angle jitter; `rng`, a NumPy generator.
- **Outputs:** a tuple `(X, y)`. `X` is a float array with shape `(n_per_class * n_classes, 2)`. `y` is an **integer** array with shape `(n_per_class * n_classes,)`. Rows are grouped by class: all of class 0 first, then class 1, and so on.
- **Recipe:** compute `radius = np.linspace(0.05, 1.0, n_per_class)` once. Then, for each class `k` in order `0, 1, …`, draw that class's angle noise with **one** call, `rng.normal(0.0, noise, n_per_class)`, compute `angle = 2 * np.pi * k / n_classes + 4.0 * radius + noise_draw`, and build the class's points from `radius * np.cos(angle)` and `radius * np.sin(angle)`. Its labels are `n_per_class` copies of `k`.

Hints: collect each class's points and labels in two lists and join each list at the end with `np.concatenate`; `np.column_stack([a, b])` turns two `(n,)` arrays into one `(n, 2)` array; `np.full(n, k)` makes `n` copies of the whole number `k`. Draw the noise exactly as described, so that your random numbers match the reference's.

Type it rather than pasting it: typing is how the shapes and indices sink in. The collapsed cell below holds a reference version, `make_spirals_reference`; open it only when you are stuck.

In [ ]:
def make_spirals_reference(n_per_class, n_classes, noise, rng):
    """Interleaved spiral arms in 2-D, one arm per class.

    Returns X with shape (n_per_class * n_classes, 2) and integer labels y with
    shape (n_per_class * n_classes,), grouped by class: class 0 first, then 1, ...
    """
    radius = np.linspace(0.05, 1.0, n_per_class)
    points, labels = [], []
    for k in range(n_classes):
        angle = 2 * np.pi * k / n_classes + 4.0 * radius + rng.normal(0.0, noise, n_per_class)
        points.append(np.column_stack([radius * np.cos(angle), radius * np.sin(angle)]))
        labels.append(np.full(n_per_class, k))
    return np.concatenate(points), np.concatenate(labels)

In [ ]:
def make_spirals(n_per_class, n_classes, noise, rng):
    """Return (X, y): interleaved spiral arms, X (n_per_class * n_classes, 2), y integer labels."""
    # ✍️ Your code here (about eight lines)
    raise NotImplementedError

In [ ]:
# A generator moves on every time it is used, so if your function and the reference shared one,
# they would receive different random numbers. The check gives each of them a fresh generator
# with the same seed, and keeps your make_spirals if the results match.
def yours_with_seed(n_per_class, n_classes, noise, seed):
    return make_spirals(n_per_class, n_classes, noise, np.random.default_rng(seed))


def reference_with_seed(n_per_class, n_classes, noise, seed):
    return make_spirals_reference(n_per_class, n_classes, noise, np.random.default_rng(seed))


checked = practice("make_spirals", yours_with_seed, reference_with_seed,
                   cases=[(5, 2, 0.0, 0), (50, 3, 0.2, 1), (1, 4, 0.1, 2)])
make_spirals = make_spirals if checked is yours_with_seed else make_spirals_reference

**Expected behaviour.** ✅ once your points, labels and noise all match. The first case has no noise at all; if only that one passes, look at how you draw the noise. The last case has a single point per class: `np.linspace(0.05, 1.0, 1)` is just `[0.05]`, so each class contributes one point near the centre.

The next cell makes the dataset used for the rest of the course, 100 points for each of 3 classes with noise 0.2, and checks that the labels are whole numbers, which later lessons need when they use labels to pick out entries of arrays.

In [ ]:
X, y = make_spirals(100, 3, 0.2, np.random.default_rng(0))
if not np.issubdtype(y.dtype, np.integer):
    print(f"❌ make_spirals: y has dtype {y.dtype}, but labels must be whole numbers (use np.full(n, k)).")
    print("   The rest of the notebook uses the reference make_spirals.")
    make_spirals = make_spirals_reference
    X, y = make_spirals(100, 3, 0.2, np.random.default_rng(0))
print("X:", X.shape, X.dtype, "   y:", y.shape, y.dtype, "   classes:", np.unique(y))
print("first three points:\n", np.round(X[:3], 3))
print("first and last labels:", y[:3], "...", y[-3:])

### Plot the spirals

A scatter plot puts each point at its coordinates. The helper below gives every class its own colour *and* marker shape, so the classes stay distinguishable for colour-blind readers and in black-and-white print, and it uses equal scales on both axes so that the spirals are not squashed.

In [ ]:
CLASS_COLOURS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6"]
CLASS_MARKERS = ["o", "s", "^", "D", "v", "P"]


def plot_points(X, y, ax, title=""):
    """Scatter 2-D points with one colour and one marker per class, and a legend."""
    for k in np.unique(y):
        chosen = y == k
        ax.scatter(X[chosen, 0], X[chosen, 1], s=24, c=CLASS_COLOURS[k % 6], marker=CLASS_MARKERS[k % 6],
                   edgecolors="white", linewidths=0.5, label=f"class {k}", zorder=3)
    ax.set_aspect("equal")
    ax.grid(True, color="#e4e3df", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_title(title, fontsize=10)
    ax.legend(loc="upper right", fontsize=8, framealpha=0.85)


fig, ax = plt.subplots(figsize=(5, 5))
plot_points(X, y, ax, "Three interleaved spirals, 100 points per class")
plt.show()

In [ ]:
# 🧪 Try it: change the number of classes and the noise, then run the cell again.
# What does noise = 0.0 look like? And noise = 0.6? (Up to 6 classes get their own colour and marker.)
n_classes_try, noise_try = 3, 0.2
X_try, y_try = make_spirals(100, n_classes_try, noise_try, np.random.default_rng(0))
fig, ax = plt.subplots(figsize=(4.5, 4.5))
plot_points(X_try, y_try, ax, f"{n_classes_try} classes, noise {noise_try}")
plt.show()

## 9. Optional: the same data as a table

pandas is a library for tables. A `DataFrame` shows the dataset with labelled columns, which makes quick summaries easy: how many points each class has, and where their averages lie. This cell is optional and tagged `data`: nothing later in the notebook uses it, and it prints a short note instead if pandas is not installed.

In [ ]:
try:
    import pandas as pd
except ImportError:
    pd = None
    print("pandas is not installed here; this optional cell has nothing to show.")

if pd is not None:
    table = pd.DataFrame({"x1": X[:, 0], "x2": X[:, 1], "label": y})
    print(table.head())
    print()
    print(table.groupby("label").agg(points=("x1", "size"), x1_mean=("x1", "mean"), x2_mean=("x2", "mean")).round(3))
    print()
    print(table[["x1", "x2"]].describe().round(3))

## 10. Put it together

This cell rebuilds everything from fixed numbers, so it does not depend on anything you changed in the 🧪 cells. It generates the course dataset with your `make_spirals`, standardises it with your `standardize`, checks your `matmul_loops` and `dot_loop` against NumPy on the result, and confirms that the same seed reproduces the same data. If an `assert` fails, its message says what went wrong.

In [ ]:
X_all, y_all = make_spirals(100, 3, 0.2, np.random.default_rng(0))
X_scaled, feature_mean, feature_std = standardize(X_all)

assert X_all.shape == (300, 2) and y_all.shape == (300,), "make_spirals should return X (300, 2) and y (300,)"
assert np.issubdtype(y_all.dtype, np.integer), "the labels should be whole numbers"
assert np.array_equal(np.bincount(y_all), [100, 100, 100]), "each class should have 100 points"
assert np.allclose(X_scaled.mean(axis=0), 0.0) and np.allclose(X_scaled.std(axis=0), 1.0), \
    "standardised columns should have mean 0 and standard deviation 1"
assert np.allclose(X_scaled * feature_std + feature_mean, X_all), "the scaling should be reversible"

W_demo = np.random.default_rng(5).normal(0.0, np.sqrt(2 / 2), size=(2, 4))
assert np.allclose(matmul_loops(X_scaled[:20], W_demo), X_scaled[:20] @ W_demo), "matmul_loops should agree with @"
assert np.isclose(dot_loop(X_scaled[0], W_demo[:, 0]), X_scaled[0] @ W_demo[:, 0]), "dot_loop should agree with @"

X_again, y_again = make_spirals(100, 3, 0.2, np.random.default_rng(0))
assert np.array_equal(X_again, X_all) and np.array_equal(y_again, y_all), "the same seed should give the same data"

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.6))
plot_points(X_all, y_all, axes[0], "Raw coordinates")
plot_points(X_scaled, y_all, axes[1], "Standardised coordinates")
plt.tight_layout()
plt.show()
print("🎉 All checks passed: your functions build, scale and reproduce the course dataset.")

## 11. Exercises

These go a step beyond the type-alongs. The first three have a stub, a collapsed reference and a check, as before; the fourth is a question to answer in your own words.

### Exercise 1: scale new data with the training statistics

When a trained model meets new data, the new data must be standardised with the mean and standard deviation of the *training* data. Using its own statistics would shift and stretch it differently from what the model learned, and a single new example would have a standard deviation of 0.

Write `standardize_with(X_new, mean, std)`, which returns `(X_new - mean) / std`. `X_new` has shape `(batch, features)`; `mean` and `std` have shape `(features,)`.

In [ ]:
def standardize_with_reference(X_new, mean, std):
    """Scale new data with statistics computed on the training data."""
    return (X_new - mean) / std

In [ ]:
def standardize_with(X_new, mean, std):
    """Return (X_new - mean) / std, using the training mean and std of shape (features,)."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
standardize_with = practice("standardize_with", standardize_with, standardize_with_reference,
                            cases=[(people[:1], people_mean, people_sd),
                                   (np.array([[170.0, 0.5], [150.0, 0.1]]), people_mean, people_sd)])

### Exercise 2: one-hot labels

Later lessons compare a network's probabilities, one column per class, with the labels. For that, each label becomes a **one-hot** row: a row of zeros with a single 1 in the column of its class. Label 2 out of 3 classes becomes `[0, 0, 1]`.

Write `one_hot(y, n_classes)`: `y` is an integer array of shape `(batch,)`, and the result is a float array of shape `(batch, n_classes)`. Hint: row `k` of the identity matrix `np.eye(n_classes)` is the one-hot row for class `k`, and indexing with an array of labels picks several rows at once.

In [ ]:
def one_hot_reference(y, n_classes):
    """One-hot rows for integer labels y (batch,) -> (batch, n_classes)."""
    return np.eye(n_classes)[y]

In [ ]:
def one_hot(y, n_classes):
    """Return a (batch, n_classes) array with a single 1.0 per row, in the column given by y."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
one_hot = practice("one_hot", one_hot, one_hot_reference,
                   cases=[(np.array([0, 2, 1]), 3),
                          (np.array([1, 1]), 2),
                          (np.array([3]), 5)])

### Exercise 3: all pairwise distances with broadcasting

Write `pairwise_sq_distances(P, Q)` for `P` with shape `(n, d)` and `Q` with shape `(m, d)`. It returns the `(n, m)` array whose entry `[i, j]` is the squared distance between point `P[i]` and point `Q[j]`, that is $\sum_k (P_{ik} - Q_{jk})^2$, **without any Python loop**.

Hint: `P[:, np.newaxis, :]` has shape `(n, 1, d)` and `Q[np.newaxis, :, :]` has shape `(1, m, d)`. Their difference broadcasts to `(n, m, d)`; square it and sum over the last axis.

In [ ]:
def pairwise_sq_distances_reference(P, Q):
    """Squared Euclidean distances between every row of P (n, d) and every row of Q (m, d)."""
    difference = P[:, np.newaxis, :] - Q[np.newaxis, :, :]
    return (difference ** 2).sum(axis=2)

In [ ]:
def pairwise_sq_distances(P, Q):
    """Return the (n, m) array of squared distances between the rows of P (n, d) and Q (m, d)."""
    # ✍️ Your code here (one or two lines)
    raise NotImplementedError

In [ ]:
pairwise_sq_distances = practice("pairwise_sq_distances", pairwise_sq_distances, pairwise_sq_distances_reference,
                                 cases=[(np.array([[0.0, 0.0], [1.0, 1.0]]), np.array([[3.0, 4.0]])),
                                        (X[:5], X[5:12]),
                                        (np.array([[1.0, 2.0, 3.0]]), np.array([[1.0, 2.0, 3.0]]))])

### Exercise 4: how fast do loops grow? (a question to answer)

Use the 🧪 cell at the end of section 5 with `size` set to 10, 20 and 40, and write down the three times.

1. By roughly what factor does the time grow each time you double `size`? Explain the factor from the three loops in `matmul_loops`.
2. NumPy's `@` performs the same number of multiplications. Give two reasons why it is still hundreds of times faster.
3. A network layer multiplies a batch of 1,000 examples with 512 features by a `(512, 512)` weight matrix. How many multiplications is that, and roughly how long would your loops take, judging by your measurements?

## 12. Common errors and debugging

| Symptom | Likely cause | Fix |
|---|---|---|
| `ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, …` | the inner sizes of `A @ B` differ | print both shapes; transpose or reshape the array that is wrong |
| a result of shape `(n, n)` where you expected `(n,)` | broadcasting a `(n,)` array against a `(n, 1)` array | flatten the column with `.ravel()`; print shapes after each step |
| `IndexError: index 3 is out of bounds for axis 0 with size 3` | a loop runs one step too far, or two indices are swapped between `A` and `B` | check which axis each loop variable walks along |
| `RuntimeWarning: invalid value encountered in divide`, then `nan` | dividing by a standard deviation of 0 | replace zeros before dividing, as `standardize` does |
| a different dataset on every run | a generator created without a seed, or calls to the global `np.random.normal` | create `rng = np.random.default_rng(seed)` once and pass it into functions |
| `IndexError: arrays used as indices must be of integer (or boolean) type` | labels stored as floats | build labels from integers, for example with `np.full(n, k)`, or convert with `y.astype(int)` |
| `NameError: name 'np' is not defined` | the setup cell has not run in this session | run the setup cell first, or use Run All |

The fastest debugging tool in this course is to print the shapes. The next cell triggers two of these errors on purpose and prints their messages.

In [ ]:
try:
    np.ones((3, 2)) @ np.ones((3, 2))
except ValueError as error:
    print("ValueError:", error)

float_labels = np.array([0.0, 2.0, 1.0])      # labels stored as floats by mistake
try:
    np.eye(3)[float_labels]
except IndexError as error:
    print("IndexError:", error)
print("after converting to integers:\n", np.eye(3)[float_labels.astype(int)])

## Summary

- A function turns inputs into an output and should raise an error on bad input; a class bundles data with methods. This course keeps parameters in dictionaries and lists, a layer being `{"W": ..., "b": ...}`, and keeps the arithmetic in small, tested functions.
- A NumPy array has one `dtype` and a fixed `shape`. Here `X` is `(batch, features)` and `y` is `(batch,)` integer labels. `axis=0` works down the rows, giving one result per feature; `axis=1` works across the columns, giving one result per example.
- Broadcasting stretches axes of size 1. That is how one bias vector is added to a whole batch, and also how `(n,) - (n, 1)` silently becomes `(n, n)`.
- Multiplying `(n, m)` by `(m, p)` gives `(n, p)`. Your three loops and `@` agree, but `@` is hundreds to thousands of times faster.
- `np.random.default_rng(seed)` makes experiments reproducible. Random weights need a scale near $\sqrt{1/n_\text{in}}$ to keep a layer's outputs at the scale of its inputs.
- Standardising rescales each feature to mean 0 and standard deviation 1, with statistics that you keep and reuse for new data.
- `make_spirals` generates this course's dataset: classes that no straight line can separate.

## Next

**Lesson 2: Build a neuron.** You will write the two most common activation functions, `sigmoid` and `relu`, and the neuron itself, $a = f(w \cdot x + b)$, first for one example and then for a whole batch. You will set its weights by hand so that it computes the logic functions AND and OR, draw the straight line it uses to decide, and then search thousands of weight settings to see that no single neuron can compute XOR: the reason networks need layers.